# 05 — Full integration test

Updated: 2026-11-05.

After notebook 02, run these cells to test every deployed model in this namespace.
For each model: preliminary training with held-out testing → verify metadata →
full training with held-out testing → approve the exact full model → batch inference.
Two models produce **six executions** with the combined training pipeline. An older
three-pipeline publication still runs its original **eight-execution** matrix.

The default launch runs in a separate background process. You can close this
notebook and browser; **keep the Studio space running**. Saved pipeline versions,
inputs and a frozen runner copy keep this test independent of later code edits.

A failed model does not stop the other models. Its dependent runs are marked
**Blocked**; independent routes continue, and the final result still records the failure.

This is a Development integration test: it approves the full models it creates.
It verifies the normal Processing paths, artifact identity and output files.
It does not establish model quality, production sizing or optional HPO/Transform,
drift and MLflow behavior.


## Load the deployed application

- Locate the checkout and load notebook 02's saved publication.
- Load the integration and background helpers using the notebook's Python environment.
- Read configuration only; no jobs or approvals are started here.


In [ ]:
from pathlib import Path
import importlib
import json
import sys
import pandas as pd

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "mlops-models/src/mlops_models").is_dir())
for directory in (ROOT / "notebooks/source/05_integration_test", ROOT / "mlops-models/src"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))
import integration_checks
import background_run
importlib.reload(integration_checks)
importlib.reload(background_run)
IntegrationSuite = integration_checks.IntegrationSuite

config_path = ROOT / "notebooks/config.json"
config = json.loads(config_path.read_text()) if config_path.is_file() else {}
namespace = config.get("namespace", "")
publication_path = ROOT / ".runs/source-deploy" / namespace / "publication.json"
publication = json.loads(publication_path.read_text()) if namespace and publication_path.is_file() else {}
print("Namespace:", namespace or "set NAMESPACE below")
print("Publication:", publication_path if publication else "provide PIPELINES below")


## Select inputs

- Leave MODEL_IDS and PIPELINES empty to use every model and the published pipelines.
- Leave CSV_INPUTS empty to use deployed data defaults; supply overrides if necessary.
- Generate a timestamped RUN_LABEL for a fresh test; paste a saved label to resume it.
- Keep RUN_IN_BACKGROUND enabled to return immediately after launch.

Training, test and inference data stay separate. This notebook does not create
splits. Editing config.json does not change deployed defaults: republish with
notebook 02 or supply CSV_INPUTS overrides.

Rerunning the input cell generates a new label by default. To resume, replace the
RUN_LABEL expression with the previously printed label in quotes; keep its inputs unchanged.


In [ ]:
from datetime import datetime, timezone

# New timestamped label each time this cell runs. To resume, paste the old label in quotes.
RUN_LABEL = datetime.now(timezone.utc).strftime("integration-%Y%m%d-%H%M%S-%f")
print("RUN_LABEL:", RUN_LABEL)
RUN_IN_BACKGROUND = True

MODEL_IDS = []                 # Every deployed model.
PIPELINES = {}                 # Saved training/inference pair; also accepts legacy testing.
CSV_INPUTS = {}                # Each pipeline's deployed data defaults.
# Example: CSV_INPUTS = {"usecase1": {"training": "s3://your-bucket/train.csv",
#     "prediction_test": "s3://your-bucket/test.csv",
#     "prediction_inference": "s3://your-bucket/inference.csv"}}

NAMESPACE = namespace
AWS_REGION = publication.get("region", "")
MODEL_CODE_SOURCE = "artifact" # "pipeline" tests the newly published scoring code.
WAIT_TIMEOUT_MINUTES = 480     # Bounds waiting; does not cancel existing AWS jobs.


## Inspect the execution plan

- Read and pin deployed pipeline versions and validate namespace, models and data inputs.
- Save the plan without starting jobs or changing approvals.
- If this label is already running, retain its existing state without modifying it.


In [ ]:
suite = IntegrationSuite(
    ROOT, publication, run_label=RUN_LABEL, model_ids=MODEL_IDS,
    csv_inputs=CSV_INPUTS, pipelines=PIPELINES, namespace=NAMESPACE, region=AWS_REGION,
    model_code_source=MODEL_CODE_SOURCE, timeout_minutes=WAIT_TIMEOUT_MINUTES)
plan = background_run.prepare_background(suite)
if plan is None:
    print("This test is already running. The launch cell will leave it running.")
else:
    for kind, pipeline in suite.pipelines.items():
        print(kind, pipeline["arn"], "version", pipeline["version"])
    print("Planned executions:", len(plan))
    print(pd.DataFrame(plan).to_string(index=False))
print("Saved evidence:", suite.directory)


## Launch the full test

- Start the sequence in the background and immediately print its log and result locations.
- Continue healthy models after another model fails; block only the failed model's dependent runs.
- Approve only this test's verified full models, then run batch inference (and standalone testing for an older publication).
- Repeated launches of an active label do not create a second worker.

**Keep the Studio space running.** Closing the notebook or restarting its kernel
is fine. Stopping the space interrupts the coordinator; existing AWS jobs are not
cancelled. Run this notebook with the same label to resume. The execution role
needs pipeline start/read, S3 read/list and model-package update permissions.

The overall test fails if any route or output check fails, after other eligible routes
finish. Timeout or interruption still stops the coordinator; it does not cancel AWS jobs.


In [ ]:
if plan is None:
    launch = background_run.background_status(suite.directory)
elif RUN_IN_BACKGROUND:
    launch = background_run.launch_background(suite)
else:
    result = suite.run()       # Optional foreground execution for debugging.
    launch = {"worker_status": result["status"], "state": str(suite.path),
              "log": "Foreground: see cell output"}
print("Integration test:", launch["worker_status"])
print("Log:", launch["log"])
print("State:", launch["state"])
print("Results:", suite.directory / "results.json")
print("The final cell reads saved progress; it does not launch or monitor jobs.")


## Read saved progress or final results

- Read the latest saved status and phase results without polling AWS or waiting.
- Show each route's outcome, error and blocked dependencies, even when other routes passed.
- Rerun this cell whenever you want an update; no monitoring is required.


In [ ]:
saved = background_run.background_status(suite.directory)
result = saved["result"]
print("Background worker:", saved["worker_status"])
if result:
    print("Result:", result["status"], "| elapsed minutes:", result["elapsed_minutes"])
    print(pd.DataFrame(result["executions"]).to_string(index=False))
    print("Checks:", result["passed_checks"], "passed,", result["failed_checks"], "failed")
    if result.get("error"):
        print("Failure:", result["error"])
if saved.get("worker", {}).get("error"):
    print("Worker failure:", saved["worker"]["error"])
print("Log:", saved["log"])
print("State for resume:", saved["state"])
